# SoC estimator comparison: controlled discharge runs

Runs coulomb counting (CC), OCV–SoC mapping, EKF and EKF + KCL on the same 10-sample median stream that
`proa_advisor/lib/Kalman Filter/kalman_filter.js` feeds the filter. It then finds the rest windows and builds the report figures and tables.
The EKF and the KCL corrector are ports of `kalman_filter_helper/filter.js` and `kalman_filter_helper/kcl_corrector.js`,
so no Node, `.env` or database is needed. The battery model and channel slot below replace the main/alternate switch in `.env`.

Set the folder, battery model and slot in the next cell, then run all cells.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'proa_advisor').is_dir())
MODEL_DIR = REPO_ROOT / 'proa_advisor' / 'model' / 'battery_model'
TEST_DATA = MODEL_DIR / 'test_data'
FIG_DIR = REPO_ROOT / 'final_report' / 'figures'
TABLE_DIR = REPO_ROOT / 'final_report' / 'tables'

sys.path.insert(0, str(REPO_ROOT / 'final_report'))
from soc_eval import FOLDERS    # test-data folder names live in soc_eval.py
FOLDER = TEST_DATA / FOLDERS['LiNMC controlled']
BATTERY = 'LiNMC'          # model folder under proa_advisor/model/battery_model
SLOT = 'main'              # 'main': current ch2, voltage ch6. 'alternate': current ch3, voltage ch7
SOC0_OVERRIDE = None       # None: start from the OCV table, as the server does
REF_SOC0 = 1.0             # the reference starts from full charge

# Rest-window definition (the same windows are used for every method)
I_ACTIVE = 1.0             # A, |I| above this counts as load or charge
I_REST = 0.5               # A, |I| below this counts as rest
T_VERIFY = 10              # s after disconnection before t0 (current verified zero)
MIN_REST = 5 * 60          # s, shorter rests are ignored
DVDT_LIMIT = 10e-3 / 60    # V/s, the 10 mV per minute criterion (pack voltage)
SLOPE_WIN = 60             # s, trailing window for the dV/dt fit
AVG_WIN = 2.5              # s, values at t0, t1 and rest end are means over +/- this
V_CUTOFF = 30              # V, BATT_CUTOFF_VOLTAGE in kalman_filter.js
SESSION_GAP = 60           # s, a larger time step starts a new session (unrecorded gap)
STRICT_VALIDATION = True   # stop if the data does not match the battery model
RUN_SENSITIVITY = True     # optional initial-SoC runs at the end (about a minute)

PREFIX = f'{BATTERY}_{FOLDER.name.replace(" ", "_")}'   # output file prefix
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

Plot style. The categorical colours (blue, orange, aqua, yellow) were checked for colour-vision separation and stay fixed per method in every figure.
Measured signals and the reference are drawn in neutral ink.

In [ ]:
COLORS = {'soc_cc': '#1baf7a', 'soc_ocv': '#eda100', 'soc_ekf': '#eb6834', 'soc_ekf_kcl': '#2a78d6'}
MARKERS = {'soc_cc': 'o', 'soc_ocv': 's', 'soc_ekf': '^', 'soc_ekf_kcl': 'D'}
INK, INK2, MUTED, GRID, AXIS, REST_FILL = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7', '#f0efec'

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Segoe UI', 'Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 9, 'axes.edgecolor': AXIS, 'axes.labelcolor': INK2, 'axes.titlesize': 10, 'axes.titlecolor': INK,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'xtick.labelcolor': INK2, 'ytick.labelcolor': INK2,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'lines.linewidth': 1.5, 'lines.solid_capstyle': 'round', 'legend.frameon': False, 'savefig.dpi': 200,
})


def save(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(FIG_DIR / f'{PREFIX}_{name}.{ext}', bbox_inches='tight')

## Sensor conversion and battery model
`adc_to_voltage` and `adc_to_current` are ports of `proa_advisor/lib/adc_converter.js`. `load_model` reads the same files the server loads into `MainRCMapping`.

In [ ]:
V_REF = 4.096
ADC_FS = {0: 2.5, 1: 1.25, 2: 0.625, 3: 0.3125, 4: 0.15625, 5: 2.5, 6: 1.25, 7: 0.625, 8: 0.3125}
VOLTAGE_DIVIDER_RATIO = (100 + 20) / 20


def adc_to_voltage(adc, rng):
    fs = ADC_FS[rng] * V_REF
    adc = np.asarray(adc, dtype=float)
    if rng <= 4:
        return (adc - 32768) / 32768 * fs
    return adc / 65536 * fs


def adc_to_current(adc, rng=1):
    v = adc_to_voltage(adc, rng)
    a = np.abs(v)
    return np.sign(v) * (0.90225289 * a ** 2 + 15.06345527 * a)


def load_model(battery):
    folder = MODEL_DIR / battery
    const = json.loads((folder / 'battery_constants.json').read_text())
    noise_file = folder / 'noise.json'
    noise = json.loads(noise_file.read_text()) if noise_file.exists() else {'socProcess': 1e-22, 'rcProcess': 1e-20}
    table = pd.read_csv(folder / 'battery_state.csv')
    # Monotone copy of the OCV curve for the inverse lookup (flat steps keep the lowest SoC)
    inv_ocv, first = np.unique(np.maximum.accumulate(table.OCV.to_numpy()), return_index=True)
    return {
        'name': battery,
        'Q_total': const['Q_total'],
        'sigma_v': const['sigma_v'],
        'sigma_kcl': const['sigma_kcl'],
        'noise': noise,
        'table': {c: table[c].to_numpy() for c in ['SoC', 'OCV', 'R0', 'R1', 'R2', 'C1', 'C2']},
        'inv_ocv': inv_ocv,
        'inv_soc': table.SoC.to_numpy()[first],
    }


def ocv_to_soc(v, model):
    """OCV-SoC mapping method: SoC interpolated from voltage on the monotone OCV curve."""
    return np.interp(v, model['inv_ocv'], model['inv_soc'])


def soc_to_ocv(soc, model):
    return np.interp(soc, model['table']['SoC'], model['table']['OCV'])


def nearest_ocv_soc(v, model):
    """Initial SoC as the server picks it: the table row with the closest OCV (getBatteryRC_OCV)."""
    t = model['table']
    return t['SoC'][np.argmin(np.abs(t['OCV'] - v))]


model = load_model(BATTERY)
print(f"{BATTERY}: Q_total {model['Q_total']:.2f} Ah, OCV {model['table']['OCV'].min():.2f}-{model['table']['OCV'].max():.2f} V, "
      f"sigma_v {model['sigma_v']:.2e} V, sigma_kcl {model['sigma_kcl']:.2e}")

## Load the run
Every CSV in the folder is read in name order. A new file or a time step over `SESSION_GAP` starts a new session; that gap was not recorded,
so it is not integrated and no rest window may span it. `detect_and_correct_flips` and `window_median` mirror `onNewSample` in `kalman_filter.js`.

In [ ]:
SLOTS = {'main': ('I_b1', 'V_b1'), 'alternate': ('I_b2', 'V_b2')}
DB_EXPORT_COLUMNS = {'time_diff': 'timediff_us', **{f'adcReading{i}': f'ch{i}' for i in range(8)}}
CURRENT_FLIP_THRESHOLD = 5.0


def read_test_file(path):
    df = pd.read_csv(path)
    if 'timediff_us' not in df.columns:
        df = df.rename(columns=DB_EXPORT_COLUMNS)   # SOCSensor export: id, timestamp, run_id, time_diff, adcReading0..7
    return df[['timediff_us'] + [f'ch{i}' for i in range(8)]].astype(float)


def detect_and_correct_flips(df):
    """Vectorised detectAndCorrectFlips: cutoff zeroing, sticky MPPT flip, battery sign flip; drops rows the server skips."""
    for i, v in (('I_b1', 'V_b1'), ('I_b2', 'V_b2')):
        low = df[v] < V_CUTOFF
        df.loc[low, [i, v]] = 0.0

    mppt = df.I_mppt.to_numpy()
    flipped_before = np.r_[False, np.maximum.accumulate(mppt < -CURRENT_FLIP_THRESHOLD)[:-1]]
    mppt = np.where(flipped_before, -mppt, mppt)
    df['I_mppt'] = np.where(mppt < -CURRENT_FLIP_THRESHOLD, -mppt, mppt)

    b1, b2, load, mppt = df.I_b1, df.I_b2, df.I_load, df.I_mppt
    violation = b1 + b2 - load + mppt > 2
    flip1 = violation & (-b1 + b2 - load + mppt < 1)
    flip2 = violation & ~flip1 & (b1 - b2 - load + mppt < 1)
    df.loc[flip1, 'I_b1'] = -df.loc[flip1, 'I_b1']
    df.loc[flip2, 'I_b2'] = -df.loc[flip2, 'I_b2']
    keep = ~violation | flip1 | flip2
    if (~keep).any():
        print(f'Skipped {(~keep).sum()} samples with an uncorrectable KCL violation (as the server does)')
    return df[keep].reset_index(drop=True)


def load_run(folder, slot='main'):
    """Raw samples in amps and volts with a session id. I_bat and V_bat are the analysed battery."""
    frames, next_session = [], 0
    for file in sorted(Path(folder).glob('*.csv')):
        df = read_test_file(file)
        dt = df.timediff_us.to_numpy() / 1e6
        new_session = np.r_[True, dt[1:] > SESSION_GAP]
        session = next_session + np.cumsum(new_session) - 1
        next_session = session[-1] + 1
        frames.append(pd.DataFrame({
            'file': file.name,
            'session': session,
            'dt': np.where(new_session, 0.0, dt),
            'gap_s': np.where(new_session, dt, 0.0),
            'I_mppt': adc_to_current(df.ch0),
            'I_load': adc_to_current(df.ch1),
            'I_b1': adc_to_current(df.ch2),
            'I_b2': adc_to_current(df.ch3),
            'V_b1': adc_to_voltage(df.ch6, 5) * VOLTAGE_DIVIDER_RATIO,
            'V_b2': adc_to_voltage(df.ch7, 5) * VOLTAGE_DIVIDER_RATIO,
        }))
    raw = detect_and_correct_flips(pd.concat(frames, ignore_index=True))
    i_col, v_col = SLOTS[slot]
    raw['I_bat'], raw['V_bat'] = raw[i_col], raw[v_col]
    raw['t'] = raw.dt.cumsum()
    return raw


def window_median(raw, n=10):
    """Median of each n-sample block within a session, dt = block sum (the stream updateFilter receives)."""
    block = raw.groupby('session').cumcount() // n
    groups = raw.groupby([raw.session, block], sort=True)
    cols = ['I_mppt', 'I_load', 'I_b1', 'I_b2', 'I_bat', 'V_bat']
    win = groups[cols].median()
    win['dt'] = groups.dt.sum()
    win = win.reset_index(level=0).reset_index(drop=True)
    win['t'] = win.dt.cumsum()
    return win


raw = load_run(FOLDER, SLOT)
win = window_median(raw)
V_INIT = raw.V_bat[raw.V_bat > V_CUTOFF].iloc[0]

sessions = raw.groupby('session').agg(file=('file', 'first'), gap_before_s=('gap_s', 'first'),
                                      start_h=('t', 'first'), end_h=('t', 'last'), samples=('t', 'size'))
sessions[['start_h', 'end_h']] /= 3600
print(f'{len(raw):,} samples -> {len(win):,} filter steps, {win.t.iloc[-1] / 3600:.2f} h recorded')
print(sessions.round(2).to_string())

## Estimators
Sign convention: positive current is discharge (as in `filter.js`).

- **REF**: coulomb counting from `REF_SOC0` (full charge), with `Q_total` and the raw battery current. It is the reference, not a method.
- **Coulomb counting**: the same integration, but it starts from the OCV table like the EKF (or from `SOC0_OVERRIDE`).
- **OCV mapping**: SoC looked up from the terminal voltage at every step. It has no memory and no IR compensation.
- **EKF**: `Battery2RCEKF` fed the raw battery current.
- **EKF + KCL**: `Battery2RCEKF` fed the current after `CurrentKCLCorrector` (the algorithm the server runs).

The ports keep the server's behaviour: the RC row comes from the previous corrected SoC, dOCV/dSoC uses the next table row,
SoC is clamped after predict and correct, and the KCL measurement noise is `sigma_kcl` itself (not squared).
At a session break the RC voltages reset to 0, because the battery has rested for days; SoC and covariance carry over.

In [ ]:
KCL_CURRENT_THRESHOLD = 0.05
METHODS = {'soc_cc': 'Coulomb counting', 'soc_ocv': 'OCV mapping', 'soc_ekf': 'EKF', 'soc_ekf_kcl': 'EKF + KCL'}


class EKF2RC:
    """Port of Battery2RCEKF (kalman_filter_helper/filter.js). State: [soc, vrc1, vrc2]."""

    def __init__(self, model, soc0):
        self.tab = model['table']
        self.n = len(self.tab['SoC'])
        self.Qn = model['Q_total'] * 3600
        self.q_soc = model['noise']['socProcess']
        self.q_rc = model['noise']['rcProcess']
        self.r = model['sigma_v'] ** 2
        self.x = np.array([soc0, 0.0, 0.0])
        self.P = np.diag([1e-8, 1e-8, 1e-8])
        self.row = self._row(soc0)

    def _row(self, soc):
        # getBatteryRC_SoC(soc, table, 2): rc_values[0] is row Math.round(soc * 1e5), rc_values[1] the next row
        if 1 - soc < 1e-5:
            return self.n - 2
        if soc < 1e-5:
            return 0
        return int(np.floor(soc * 100 * 1000 + 0.5))

    def reset_rc(self):
        self.x[1:] = 0.0

    def update(self, dt, v, i):
        tab, k = self.tab, self.row
        r0, r1, r2, c1, c2, ocv = (tab[c][k] for c in ('R0', 'R1', 'R2', 'C1', 'C2', 'OCV'))
        a1, a2 = np.exp(-dt / (r1 * c1)), np.exp(-dt / (r2 * c2))

        soc, v1, v2 = self.x
        x = np.array([min(max(soc - i * dt / self.Qn, 0.0), 1.0),
                      a1 * v1 + r1 * (1 - a1) * i,
                      a2 * v2 + r2 * (1 - a2) * i])
        F = np.diag([1.0, a1, a2])
        P = F @ self.P @ F.T + np.diag([self.q_soc * dt, self.q_rc * dt, self.q_rc * dt])

        H = np.array([(tab['OCV'][k + 1] - ocv) / (tab['SoC'][k + 1] - tab['SoC'][k]), -1.0, -1.0])
        K = P @ H / (H @ P @ H + self.r)
        x = x + K * (v - (ocv - x[1] - x[2] - r0 * i))
        x[0] = min(max(x[0], 0.0), 1.0)

        self.x, self.P = x, P - np.outer(K, H @ P)
        self.row = self._row(x[0])
        return x[0]


class KCLCorrector:
    """Port of CurrentKCLCorrector (kalman_filter_helper/kcl_corrector.js). State: biases of [load, charge, bat1, bat2]."""
    PROCESS = np.array([1e-12, 1e-12, 1e-18, 1e-18])
    SIGN = np.array([-1.0, 1.0, 1.0, 1.0])

    def __init__(self, r_kcl):
        self.r = r_kcl
        self.b = np.zeros(4)
        self.P = np.diag([1e-4, 1e-4, 1e-5, 1e-5])

    def update(self, dt, currents, active):
        raw_kcl = currents[0] - currents[1] - currents[2] - currents[3]
        P = self.P + np.diag(np.where(active, self.PROCESS * dt, 0.0))
        H = self.SIGN * active
        K = P @ H / (H @ P @ H + self.r)
        self.b = self.b + K * (0.0 - (raw_kcl + self.SIGN @ self.b))   # pseudo-measurement z = 0
        self.P = P - np.outer(K, H @ P)

        corrected = np.where(active, currents - self.b, 0.0)
        corrected_kcl = corrected[0] - corrected[1] - corrected[2] - corrected[3]
        if corrected_kcl > raw_kcl:   # over-correction guard: fall back to raw currents
            corrected = np.where(active, currents, 0.0)
            corrected_kcl = raw_kcl
        return corrected, raw_kcl, corrected_kcl


def run_estimators(win, model, v_init, soc0=None):
    dt, v, i_bat = win.dt.to_numpy(), win.V_bat.to_numpy(), win.I_bat.to_numpy()
    currents = win[['I_load', 'I_mppt', 'I_b1', 'I_b2']].to_numpy()
    session = win.session.to_numpy()
    bat = 2 if SLOT == 'main' else 3

    soc_init = nearest_ocv_soc(v_init, model) if soc0 is None else soc0
    ekf, ekf_kcl = EKF2RC(model, soc_init), EKF2RC(model, soc_init)
    kcl = KCLCorrector(model['sigma_kcl'])

    n = len(win)
    i_kcl, kcl_raw, kcl_corr, bias, bias_load = (np.zeros(n) for _ in range(5))
    soc_ekf, soc_ekf_kcl = np.full(n, np.nan), np.full(n, np.nan)
    for k in range(n):
        if k and session[k] != session[k - 1]:
            ekf.reset_rc()
            ekf_kcl.reset_rc()
        c = currents[k]
        active = np.array([c[0] > KCL_CURRENT_THRESHOLD, c[1] > KCL_CURRENT_THRESHOLD,
                           abs(c[2]) > KCL_CURRENT_THRESHOLD, abs(c[3]) > KCL_CURRENT_THRESHOLD])
        corrected, kcl_raw[k], kcl_corr[k] = kcl.update(dt[k], c, active)
        i_kcl[k], bias[k], bias_load[k] = corrected[bat], kcl.b[bat], kcl.b[0]
        if v[k] > V_CUTOFF and dt[k] > 0:
            soc_ekf[k] = ekf.update(dt[k], v[k], i_bat[k])
            soc_ekf_kcl[k] = ekf_kcl.update(dt[k], v[k], i_kcl[k])
        else:
            soc_ekf[k], soc_ekf_kcl[k] = ekf.x[0], ekf_kcl.x[0]

    Qn = model['Q_total'] * 3600
    return pd.DataFrame({
        't': win.t, 'dt': dt, 'session': session, 'I_bat': i_bat, 'I_kcl': i_kcl, 'V': v,
        'soc_ref': REF_SOC0 - np.cumsum(i_bat * dt) / Qn,
        'soc_cc': soc_init - np.cumsum(i_bat * dt) / Qn,
        'soc_ocv': np.where(v > V_CUTOFF, ocv_to_soc(v, model), np.nan),
        'soc_ekf': soc_ekf,
        'soc_ekf_kcl': soc_ekf_kcl,
        'soc_cc_kcl': soc_init - np.cumsum(i_kcl * dt) / Qn,
        'kcl_raw': kcl_raw, 'kcl_corrected': kcl_corr, 'bias_bat': bias, 'bias_load': bias_load,
    })


est = run_estimators(win, model, V_INIT, SOC0_OVERRIDE)
print(f'Initial SoC from OCV table at {V_INIT:.3f} V: {100 * nearest_ocv_soc(V_INIT, model):.2f} %')
print((100 * est[['soc_ref', *METHODS]].iloc[[0, -1]]).round(2).to_string())

## Rest windows
For each rest block (|I| < `I_REST` for at least `MIN_REST`, inside one session):
- **t_disc**: the first rest step after the load is disconnected.
- **t0** = t_disc + `T_VERIFY`, once the current is verified zero. The disconnection jump itself is excluded.
- **t1**: the first time at least `SLOPE_WIN` after t0 when the least-squares slope of V over the trailing `SLOPE_WIN` is ≤ 10 mV/min.
  If that never happens, t1 is the rest end and `criterion_met` is False.

ΔSoC is in percentage points (pp). As the guidance says, describe it as rest-period variation, not SoC error.

In [ ]:
def _trailing_slope(t, v, window):
    """Least-squares dV/dt over the trailing `window` seconds at every sample (NaN until a full window exists)."""
    c = lambda a: np.r_[0.0, np.cumsum(a)]
    St, Sv, Stt, Stv = c(t), c(v), c(t * t), c(t * v)
    hi = np.arange(1, len(t) + 1)
    lo = np.searchsorted(t, t - window, side='right')
    n = hi - lo
    st, sv, stt, stv = St[hi] - St[lo], Sv[hi] - Sv[lo], Stt[hi] - Stt[lo], Stv[hi] - Stv[lo]
    with np.errstate(invalid='ignore', divide='ignore'):
        slope = (n * stv - st * sv) / (n * stt - st ** 2)
    slope[t - t[0] < window] = np.nan
    return slope


def _mean_near(t, y, tc, lo, hi):
    mask = (t >= max(tc - AVG_WIN, lo)) & (t <= min(tc + AVG_WIN, hi))
    return np.nanmean(y[mask])


def find_rests(est):
    rests = []
    for session, g in est.groupby('session'):
        t, i, v = g.t.to_numpy(), g.I_bat.to_numpy(), g.V.to_numpy()
        is_rest = (np.abs(i) < I_REST) & (v > V_CUTOFF)
        edges = np.flatnonzero(np.diff(np.r_[0, is_rest.astype(int), 0]))
        for s, e in zip(edges[::2], edges[1::2]):      # rest steps are s .. e-1
            t_disc, t_end = t[s], t[e - 1]
            if t_end - t_disc < MIN_REST:
                continue
            active = np.flatnonzero(np.abs(i[:s]) > I_ACTIVE)
            if not len(active):
                continue                                 # no load before this rest in the session
            run_start = active[np.flatnonzero(np.diff(np.r_[-2, active]) > 1)[-1]]
            kind = 'after_discharge' if i[run_start:active[-1] + 1].mean() > 0 else 'after_charge'

            t0 = t_disc + T_VERIFY
            seg = slice(np.searchsorted(t, t0), e)
            slope = _trailing_slope(t[seg] - t0, v[seg], SLOPE_WIN)
            ok = np.flatnonzero(np.abs(slope) <= DVDT_LIMIT)
            t1 = t[seg][ok[0]] if len(ok) else t_end
            rests.append({'session': session, 'type': kind, 't_disc': t_disc, 't0': t0, 't1': t1, 't_end': t_end,
                          'criterion_met': bool(len(ok)), 'I_before': i[run_start:active[-1] + 1].mean()})
    rests = pd.DataFrame(rests)
    rests.insert(0, 'rest', np.arange(1, len(rests) + 1))
    return rests


def rest_deltas(est, rests):
    """One row per rest and method: signed ΔSoC (pp) from t0 to t1 and to the rest end, with ΔV and durations."""
    t = est.t.to_numpy()
    rows = []
    for r in rests.itertuples():
        at = lambda col, tc: _mean_near(t, est[col].to_numpy(), tc, r.t_disc, r.t_end)
        v0, v1, v_end = at('V', r.t0), at('V', r.t1), at('V', r.t_end)
        for col, name in METHODS.items():
            s0, s1, s_end = at(col, r.t0), at(col, r.t1), at(col, r.t_end)
            dur_h = (r.t1 - r.t0) / 3600
            rows.append({
                'rest': r.rest, 'session': r.session, 'type': r.type, 'method': name,
                'soc_ref_t0_pct': 100 * at('soc_ref', r.t0),
                'duration_min': 60 * dur_h, 'rest_length_min': (r.t_end - r.t_disc) / 60,
                'criterion_met': r.criterion_met,
                'dV_V': v1 - v0, 'dV_end_V': v_end - v0, 'V_end': v_end,
                'dsoc_pp': 100 * (s1 - s0),
                'dsoc_per_h_pp': 100 * (s1 - s0) / dur_h if dur_h > 0 else np.nan,
                'dsoc_end_pp': 100 * (s_end - s0),
                'v_falls': r.type == 'after_discharge' and v_end < v0,
            })
    return pd.DataFrame(rows)


def summary_table(deltas, battery):
    rows = []
    for (kind, method), g in deltas.groupby(['type', 'method'], sort=False):
        rows.append({
            'Battery': battery, 'Rest type': kind.replace('_', ' '), 'Number of rests': len(g), 'Method': method,
            'Median |ΔSoC| t0→t1 (pp)': g.dsoc_pp.abs().median(),
            'Largest |ΔSoC| t0→t1 (pp)': g.dsoc_pp.abs().max(),
            'Rest duration t0→t1 (min)': f'{g.duration_min.min():.1f}–{g.duration_min.max():.1f}',
            'Median |ΔSoC|/h (pp/h)': g.dsoc_per_h_pp.abs().median(),
            'Median |ΔSoC| t0→end (pp)': g.dsoc_end_pp.abs().median(),
        })
    return pd.DataFrame(rows)


def to_markdown(df, digits=3):
    cells = df.map(lambda x: f'{x:.{digits}f}' if isinstance(x, float) else str(x))
    lines = ['| ' + ' | '.join(df.columns) + ' |', '|' + '---|' * len(df.columns)]
    lines += ['| ' + ' | '.join(row) + ' |' for row in cells.to_numpy()]
    return '\n'.join(lines)


rests = find_rests(est)
deltas = rest_deltas(est, rests)
per_rest = deltas.pivot_table(index=['rest', 'session', 'type', 'soc_ref_t0_pct', 'duration_min', 'rest_length_min',
                                     'criterion_met', 'dV_V', 'v_falls'], columns='method', values='dsoc_pp', sort=False)
print(f'{len(rests)} rest windows')
print(per_rest[list(METHODS.values())].round(3).to_string())

## Check that the data matches the battery model
Before comparing methods, confirm that the run belongs to the configured battery. Each battery model in `proa_advisor/model/battery_model` is
scored on the same rest-end voltages: REF SoC is computed with that model's `Q_total`, and the RMS gap to its OCV curve is measured.
Checks: (1) every rest-end voltage lies inside the configured model's OCV range; (2) the discharged Ah is within 10 % of `Q_total`;
(3) the configured model fits the rest-end voltages better than any other model; (4) its RMS gap is under 0.5 V.

In [ ]:
rest_end = deltas.drop_duplicates('rest').set_index('rest')
t_end = rests.set_index('rest').t_end
discharged_ah = np.sum(est.I_bat * est.dt) / 3600

fits = []
for folder in sorted(p for p in MODEL_DIR.iterdir() if (p / 'battery_state.csv').exists()):
    m = load_model(folder.name)
    soc_end = REF_SOC0 - np.interp(t_end, est.t, np.cumsum(est.I_bat * est.dt)) / (m['Q_total'] * 3600)
    resid = rest_end.V_end - soc_to_ocv(soc_end, m)
    fits.append({'model': m['name'], 'Q_total_Ah': m['Q_total'],
                 'OCV_min_V': m['table']['OCV'].min(), 'OCV_max_V': m['table']['OCV'].max(),
                 'rms_gap_V': np.sqrt(np.mean(resid ** 2)),
                 'table_rise_25_75_V': soc_to_ocv(0.75, m) - soc_to_ocv(0.25, m)})
fits = pd.DataFrame(fits).set_index('model')

soc_ref_end = 100 * np.interp(t_end, est.t, est.soc_ref)
mid = (soc_ref_end >= 25) & (soc_ref_end <= 75)
data_rise = np.polyfit(soc_ref_end[mid] / 100, rest_end.V_end[mid], 1)[0] * 0.5 if mid.sum() >= 2 else np.nan

checks = pd.DataFrame([
    ('Rest-end V inside OCV table range',
     f'{rest_end.V_end.min():.2f}-{rest_end.V_end.max():.2f} V vs {fits.loc[BATTERY, "OCV_min_V"]:.2f}-{fits.loc[BATTERY, "OCV_max_V"]:.2f} V',
     rest_end.V_end.between(fits.loc[BATTERY, 'OCV_min_V'], fits.loc[BATTERY, 'OCV_max_V']).all()),
    ('Discharged Ah within 10 % of Q_total',
     f'{discharged_ah:.2f} Ah vs {model["Q_total"]:.2f} Ah ({100 * (discharged_ah / model["Q_total"] - 1):+.1f} %)',
     abs(discharged_ah / model['Q_total'] - 1) <= 0.10),
    ('Configured model fits best', f'best fit: {fits.rms_gap_V.idxmin()}', fits.rms_gap_V.idxmin() == BATTERY),
    ('RMS gap to OCV curve < 0.5 V', f'{fits.loc[BATTERY, "rms_gap_V"]:.3f} V', fits.loc[BATTERY, 'rms_gap_V'] < 0.5),
], columns=['check', 'value', 'pass'])
print(fits.round(3).to_string())
print(f'\nRest-end voltage rise between REF SoC 25 % and 75 % in the data: {data_rise:.2f} V')
print(checks.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
soc_axis = np.linspace(0, 1, 501)
for name, color in zip(fits.index, ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']):
    ax.plot(100 * soc_axis, soc_to_ocv(soc_axis, load_model(name)), color=color, label=f'{name} OCV table')
ax.scatter(soc_ref_end, rest_end.V_end, s=36, color=INK, edgecolor='white', linewidth=1.5, zorder=3,
           label='Rest-end voltage (this run)')
ax.set(xlabel=f'REF SoC at rest end (%, {BATTERY} Q_total)', ylabel='Voltage (V)')
ax.set_title(f'{FOLDER.name}: rest-end voltage against each OCV table', loc='left')
ax.legend()
fig.tight_layout()
save(fig, 'model_check')

if STRICT_VALIDATION and not checks['pass'].all():
    raise RuntimeError(f'{FOLDER.name} does not match the {BATTERY} model; see the checks above.')

## Results table
Saved to `final_report/tables/` as CSV and Markdown. Rests flagged `v_falls` (voltage falls during a rest after discharge) need a manual look
before you use them.

In [ ]:
summary = summary_table(deltas, BATTERY)
rests.to_csv(TABLE_DIR / f'{PREFIX}_rest_windows.csv', index=False)
deltas.to_csv(TABLE_DIR / f'{PREFIX}_rest_deltas.csv', index=False)
summary.to_csv(TABLE_DIR / f'{PREFIX}_rest_summary.csv', index=False)
(TABLE_DIR / f'{PREFIX}_rest_summary.md').write_text(to_markdown(summary), encoding='utf-8')
print(to_markdown(summary))
flagged = rest_end[rest_end.v_falls].index.tolist()
print(f'\nRests where voltage falls during rest after discharge: {flagged or "none"}')

## Figures
1. **Whole run**: current, voltage and SoC, with rest blocks shaded. The open dot marks t₀ and the filled dot marks t₁.
2. **Rest close-up**: from 1 min before disconnection to the end of the rest. OCV mapping gets its own SoC panel because it moves 10–100× more than the others.
3. **Rest summary**: signed ΔSoC from t₀ to t₁ for every rest, with that rest's ΔV and duration.

In [ ]:
CURRENT_METHODS = {c: METHODS[c] for c in ('soc_cc', 'soc_ekf', 'soc_ekf_kcl')}


def label_lines(ax, marks):
    """Label vertical marker lines at the top of an axis, staggering labels that sit close together."""
    last_x, row = None, 0
    span = np.diff(ax.get_xlim())[0]
    for x, text in marks:
        row = row + 1 if last_x is not None and x - last_x < 0.06 * span else 0
        ax.annotate(text, (x, 1), xycoords=('data', 'axes fraction'), xytext=(3, -2 - 11 * row),
                    textcoords='offset points', va='top', fontsize=8, color=INK2)
        last_x = x


def plot_whole_run(est, rests):
    hours = est.t / 3600
    valid = est.V > V_CUTOFF
    fig, axes = plt.subplots(3, 1, figsize=(10, 7.5), sharex=True, gridspec_kw={'height_ratios': [1, 1, 1.4]})
    axes[0].plot(hours, est.I_bat.where(valid), color=INK2, lw=1)
    axes[0].set_ylabel('Battery current (A)\n+ = discharge')
    axes[1].plot(hours, est.V.where(valid), color=INK2, lw=1)
    axes[1].set_ylabel('Voltage (V)')
    axes[2].plot(hours, 100 * est.soc_ref, color=INK, ls=(0, (4, 3)), label='REF (CC from full)', zorder=5)
    for col, name in METHODS.items():
        axes[2].plot(hours, 100 * est[col], color=COLORS[col], label=name, zorder=2 if col == 'soc_ocv' else 3)
    axes[2].set_ylabel('SoC (%)')
    axes[2].set_xlabel('Recorded test time (h), unrecorded gaps removed')
    axes[2].legend(ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.22))

    for r in rests.itertuples():
        for ax in axes:
            ax.axvspan(r.t_disc / 3600, r.t_end / 3600, color=REST_FILL, lw=0, zorder=0)
    axes[1].scatter(rests.t0 / 3600, np.interp(rests.t0, est.t, est.V), s=28, facecolor='white', edgecolor=INK, lw=1.2,
                    zorder=4, label='t₀ (current verified zero)')
    axes[1].scatter(rests.t1 / 3600, np.interp(rests.t1, est.t, est.V), s=28, color=INK, edgecolor='white', lw=1.2,
                    zorder=4, label='t₁ (10 mV/min reached)')
    axes[1].legend(loc='lower left')
    breaks = est.t[est.session.diff() > 0] / 3600
    for ax in axes:
        for t_break in breaks:
            ax.axvline(t_break, color=MUTED, lw=1, ls=(0, (3, 3)), zorder=1)
    label_lines(axes[0], [(t, 'unrecorded gap') for t in breaks])
    axes[0].set_title(f'{BATTERY} controlled discharge ({FOLDER.name}), rest periods shaded', loc='left', pad=16)
    fig.tight_layout()
    return fig


def plot_rest_closeup(est, rests, rest_id=None):
    if rest_id is None:
        # Representative rest: the median-length one among the five nearest 50 % SoC
        cand = rest_end.assign(dist=(rest_end.soc_ref_t0_pct - 50).abs()).nsmallest(5, 'dist')
        rest_id = cand.rest_length_min.sub(cand.rest_length_min.median()).abs().idxmin()
    r = rests.set_index('rest').loc[rest_id]
    seg = est[(est.t >= r.t_disc - 60) & (est.t <= r.t_end)]
    minutes = (seg.t - r.t_disc) / 60
    marks = [(0, 'disconnect'), ((r.t0 - r.t_disc) / 60, 't₀'), ((r.t1 - r.t_disc) / 60, 't₁')]

    fig, axes = plt.subplots(4, 1, figsize=(8, 8.5), sharex=True, gridspec_kw={'height_ratios': [1.1, 0.8, 0.9, 1.2]})
    axes[0].plot(minutes, seg.V, color=INK2)
    axes[0].set_ylabel('Voltage (V)')

    axes[1].plot(minutes, seg.I_bat, color=INK2, label='Raw')
    axes[1].plot(minutes, seg.I_kcl, color=COLORS['soc_ekf_kcl'], lw=1, label='KCL corrected')
    rest_i = seg.loc[seg.t >= r.t0, ['I_bat', 'I_kcl']]
    axes[1].set_ylim(min(rest_i.min().min(), 0) - 0.2, max(rest_i.max().max(), 0) + 0.3)
    axes[1].set_ylabel('Current (A)')
    axes[1].legend(loc='upper right', ncol=2)
    axes[1].annotate(f'Load current before disconnection: {seg.I_bat[minutes < 0].mean():.1f} A (off scale)', (0, 1),
                     xycoords='axes fraction', xytext=(4, -2), textcoords='offset points', va='top', fontsize=7.5, color=INK2)

    t_all = est.t.to_numpy()
    for ax, cols in ((axes[2], {'soc_ocv': METHODS['soc_ocv']}), (axes[3], CURRENT_METHODS)):
        lo, hi = [], []
        for col, name in cols.items():
            d = 100 * (seg[col] - _mean_near(t_all, est[col].to_numpy(), r.t0, r.t_disc, r.t_end))
            ax.plot(minutes, d, color=COLORS[col], label=name)
            lo.append(d[seg.t >= r.t0].min())
            hi.append(d[seg.t >= r.t0].max())
        pad = 0.15 * max(max(hi) - min(lo), 0.02)
        ax.set_ylim(min(lo) - pad, max(hi) + pad)
        ax.set_ylabel('SoC change\nfrom t₀ (pp)')
        ax.legend(loc='upper left', ncol=len(cols))
    axes[3].set_xlabel('Minutes from load disconnection (y-axes are scaled to t₀ onwards; earlier values may be clipped)')

    for ax in axes:
        for x, _ in marks:
            ax.axvline(x, color=MUTED, lw=1, ls=(0, (3, 3)))
    label_lines(axes[0], marks)
    axes[0].set_title(f'Rest {rest_id} (session {r.session + 1}, REF SoC {rest_end.loc[rest_id, "soc_ref_t0_pct"]:.0f} % at t₀)',
                      loc='left', pad=16)
    fig.tight_layout()
    return fig, rest_id


def plot_rest_summary(deltas):
    fig, axes = plt.subplots(4, 1, figsize=(9, 8), sharex=True, gridspec_kw={'height_ratios': [1, 1.3, 0.7, 0.7]})
    for ax, cols in ((axes[0], {'soc_ocv': METHODS['soc_ocv']}), (axes[1], CURRENT_METHODS)):
        for j, (col, name) in enumerate(cols.items()):
            g = deltas[deltas.method == name]
            x = g.rest + (j - (len(cols) - 1) / 2) * 0.2
            ax.scatter(x, g.dsoc_pp, s=36, marker=MARKERS[col], color=COLORS[col], edgecolor='white', lw=1, zorder=3, label=name)
        ax.axhline(0, color=AXIS, lw=1, zorder=1)
        ax.set_ylabel('Signed ΔSoC\nt₀→t₁ (pp)')
        ax.legend(loc='upper right', ncol=len(cols))
    axes[2].scatter(rest_end.index, 1000 * rest_end.dV_V, s=30, color=INK2, edgecolor='white', lw=1, zorder=3)
    axes[2].set_ylabel('ΔV t₀→t₁\n(mV)')
    axes[3].scatter(rest_end.index, rest_end.duration_min, s=30, color=INK2, edgecolor='white', lw=1, zorder=3)
    axes[3].set_ylabel('t₁ − t₀\n(min)')
    axes[3].set_xticks(rest_end.index, [f'{i}\n{rest_end.loc[i, "soc_ref_t0_pct"]:.0f}%' for i in rest_end.index])
    axes[3].set_xlabel('Rest number (REF SoC at t₀)')
    flagged = rest_end.index[rest_end.v_falls | ~rest_end.criterion_met]
    for i in flagged:
        for ax in axes:
            ax.axvspan(i - 0.45, i + 0.45, color=REST_FILL, lw=0, zorder=0)
    note = ' (shaded: 10 mV/min not reached or voltage falls)' if len(flagged) else ''
    axes[0].set_title(f'{BATTERY}: change during each rest{note}', loc='left')
    fig.tight_layout()
    return fig


fig1 = plot_whole_run(est, rests)
save(fig1, 'whole_run')
fig2, closeup_rest = plot_rest_closeup(est, rests)
save(fig2, f'rest_{closeup_rest}_closeup')
fig3 = plot_rest_summary(deltas)
save(fig3, 'rest_summary')
plt.show()

## Optional: current-measurement consistency (KCL)
The raw KCL residual (load − charge − battery currents) should be 0. The corrector estimates a bias for each sensor so that the corrected residual is smaller.
With only a load sensor and a battery sensor active, it can observe only the *difference* between their biases, so watch whether both estimates drift together.
The rest-period mean current is the offset that CC and the EKF integrate during each rest.

In [ ]:
at_rest = (est.I_bat.abs() < I_REST) & (est.V > V_CUTOFF)
print(f'Mean battery current at rest: raw {est.I_bat[at_rest].mean():+.3f} A, KCL corrected {est.I_kcl[at_rest].mean():+.3f} A')
print(f'Mean |KCL residual|: raw {est.kcl_raw.abs().mean():.3f} A, corrected {est.kcl_corrected.abs().mean():.3f} A')
print(f'Final bias estimates: load {est.bias_load.iloc[-1]:+.3f} A, battery {est.bias_bat.iloc[-1]:+.3f} A')

hours = est.t / 3600
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(hours, est.kcl_raw, color=INK2, lw=0.8, label='Raw')
axes[0].plot(hours, est.kcl_corrected, color=COLORS['soc_ekf_kcl'], lw=0.8, label='Corrected')
axes[0].set_ylim(-1.5, 1.5)
axes[0].set_ylabel('KCL residual (A)')
axes[0].legend(loc='upper right', ncol=2)
axes[0].set_title('Residual clipped to ±1.5 A; spikes at load switching are sensor timing', loc='left')
axes[1].plot(hours, est.bias_load, color=INK2, label='Load sensor')
axes[1].plot(hours, est.bias_bat, color=COLORS['soc_ekf_kcl'], label='Battery sensor')
axes[1].set_ylabel('Bias estimate (A)')
axes[1].set_xlabel('Recorded test time (h)')
axes[1].legend(loc='upper left', ncol=2)
fig.tight_layout()
save(fig, 'kcl_consistency')
plt.show()

## Optional: initial-SoC sensitivity
Every method except OCV mapping restarts from a deliberately wrong SoC0, and each trace is plotted against REF.
OCV mapping has no memory, so it is unaffected.

In [ ]:
if RUN_SENSITIVITY:
    soc0_runs = [0.8, 0.6]
    fig, axes = plt.subplots(len(soc0_runs), 1, figsize=(10, 2.8 * len(soc0_runs)), sharex=True, sharey=True)
    for ax, soc0 in zip(axes, soc0_runs):
        r = run_estimators(win, model, V_INIT, soc0=soc0)
        for col, name in METHODS.items():
            ax.plot(r.t / 3600, 100 * (r[col] - r.soc_ref), color=COLORS[col], label=name)
        ax.axhline(0, color=AXIS, lw=1)
        ax.set_title(f'SoC0 = {100 * soc0:.0f} %', loc='left')
        ax.set_ylabel('SoC − REF (pp)')
        settle = r.t[(r.soc_ekf_kcl - r.soc_ref).abs() < 0.02].iloc[0]
        print(f'SoC0 = {100 * soc0:.0f} %: EKF + KCL within 2 pp of REF after {settle:.0f} s; final SoC − REF (pp) '
              + ', '.join(f'{n} {100 * (r[c].iloc[-1] - r.soc_ref.iloc[-1]):+.2f}' for c, n in METHODS.items()))
    axes[-1].set_xlabel('Recorded test time (h)')
    axes[0].legend(ncol=4, loc='lower center', bbox_to_anchor=(0.5, 1.12))
    fig.tight_layout()
    save(fig, 'initial_soc_sensitivity')
    plt.show()